# 06 — Construire des annotations depuis une vidéo avec un gros teacher

Ce notebook transforme une vidéo en candidat pour le futur dataset `v003`. Le gros modèle n'est pas destiné à l'iPhone : il propose seulement des annotations que nous corrigeons ensuite humainement. Le nano déjà adapté à `v002` complète ses propositions.

Pipeline :

1. conserver une image source sur quatre ;
2. faire proposer des boîtes par `yolo26x` et par notre nano fine-tuné ;
3. convertir la classe COCO `sports ball` en classe PitchLens `ball` ;
4. fusionner les deux résultats en supprimant les boîtes qui se recouvrent ;
5. enregistrer image, timestamp, confiance, teacher d'origine et boîte ;
6. revoir chaque proposition dans la même interface que pour `v002`.

Le pilote sélectionne 100 images réparties sur **toute** la vidéo. Il ne prend pas simplement les 100 premières.

In [ ]:
%matplotlib widget

from pathlib import Path
import importlib
import json
import sys

import cv2
import matplotlib.pyplot as plt
import torch
from IPython.display import Image as DisplayImage
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

PROJECT_ROOT = next(
    folder
    for folder in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (folder / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pitchlens.data.pseudo_label_video as pseudo_label_module

# Important dans un notebook : recharge le module après une correction du code.
pseudo_label_module = importlib.reload(pseudo_label_module)
combine_teacher_datasets = pseudo_label_module.combine_teacher_datasets
expected_sample_count = pseudo_label_module.expected_sample_count
prepare_video_pseudo_labels = pseudo_label_module.prepare_video_pseudo_labels

print(f"Projet : {PROJECT_ROOT}")

## 1. Paramètres de l'expérience

`PILOT = True` produit seulement 100 images, réparties du début à la fin. Après inspection, passer à `False` produira les 1 232 images espacées de quatre frames dans un autre dossier.

Le seuil `0.10` est volontairement bas : pendant l'annotation, nous préférons proposer un ballon douteux que rater un ballon en l'air. Les fausses propositions pourront être supprimées.

In [ ]:
VIDEO_PATH = PROJECT_ROOT / "data/external/videos/inference_input_yt.mp4"
LARGE_TEACHER = (
    PROJECT_ROOT
    / "models/checkpoints/teachers/yolo26x_v002_hf/highres_960/best.pt"
)
DOMAIN_TEACHER = (
    PROJECT_ROOT
    / "models/checkpoints/baseline_runs/yolo26n_v002_local_20260918_002618"
    / "train/weights/best.pt"
)

PILOT = True
FRAME_STRIDE = 4
MAX_SAMPLES = 100 if PILOT else None
# Le teacher a été continué à 960 px : c'est sa résolution de référence.
IMAGE_SIZE = 960
CONFIDENCE = 0.10
MAX_DETECTIONS = 5
SOURCE_CLASS_ID = 0  # le teacher fine-tuné ne contient que la classe ball
# Le YOLO26x à 960 px fait tomber le kernel de ce Mac 8 Go.
# Ses propositions sont donc calculées sur Hugging Face puis téléchargées ici.
LARGE_INFERENCE_BACKEND = "Hugging Face A10G"

suffix = "pilot100" if PILOT else "full"
LARGE_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data/interim"
    / f"video_teacher_{VIDEO_PATH.stem}_{suffix}_yolo26x_highres960"
)
DOMAIN_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data/interim"
    / f"video_teacher_{VIDEO_PATH.stem}_{suffix}_yolo26n_finetuned"
)
ENSEMBLE_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data/interim"
    / f"video_teacher_{VIDEO_PATH.stem}_{suffix}_ensemble_highres960"
)

capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened():
    raise FileNotFoundError(f"Vidéo illisible : {VIDEO_PATH}")
total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
fps = float(capture.get(cv2.CAP_PROP_FPS) or 30.0)
capture.release()
full_count = expected_sample_count(total_frames, FRAME_STRIDE)
selected_count = min(full_count, MAX_SAMPLES) if MAX_SAMPLES else full_count

display(Markdown(
    f"**Vidéo :** `{VIDEO_PATH.name}` — {total_frames:,} images — {total_frames / fps:.1f} s  \\n"
    f"**Sélection :** {selected_count:,} images sur {full_count:,} possibles  \\n"
    f"**Teachers :** `{LARGE_TEACHER.name}` adapté (principal) + nano adapté (complément) à {IMAGE_SIZE}px  \\n"
    f"**Gros teacher :** `{LARGE_INFERENCE_BACKEND}`  \\n"
    f"**Sortie à revoir :** `{ENSEMBLE_OUTPUT_ROOT}`"
))

## 2. Extraire et pseudo-annoter

Le grand `yolo26x` affiné sur PitchLens est prioritaire lorsqu'une boîte des deux modèles se recouvre ; le nano adapté ajoute uniquement ses propositions complémentaires. Le calcul lourd du grand modèle est effectué sur un GPU Hugging Face, puis ce notebook relit les résultats téléchargés. Il ne charge donc pas le modèle 960 px dans les 8 Go du Mac. Les résultats sont immuables : une relance ne remplace ni les images ni tes décisions.

In [ ]:
large_report_path = LARGE_OUTPUT_ROOT / "pseudo_label_report.json"
if large_report_path.exists():
    large_report = json.loads(large_report_path.read_text())
    print("Propositions yolo26x déjà présentes : aucun fichier remplacé.")
elif LARGE_INFERENCE_BACKEND == "Hugging Face A10G":
    raise RuntimeError(
        "Les propositions du gros teacher doivent d'abord être calculées sur "
        "Hugging Face : son chargement local dépasse les 8 Go du Mac."
    )
else:
    large_report = prepare_video_pseudo_labels(
        project_root=PROJECT_ROOT,
        video_path=VIDEO_PATH,
        output_root=LARGE_OUTPUT_ROOT,
        teacher_model=LARGE_TEACHER,
        frame_stride=FRAME_STRIDE,
        max_samples=MAX_SAMPLES,
        image_size=IMAGE_SIZE,
        confidence=CONFIDENCE,
        max_detections=MAX_DETECTIONS,
        source_class_id=SOURCE_CLASS_ID,
        device="cpu",
    )

domain_report_path = DOMAIN_OUTPUT_ROOT / "pseudo_label_report.json"
if domain_report_path.exists():
    domain_report = json.loads(domain_report_path.read_text())
    print("Propositions du nano adapté déjà présentes : aucun fichier remplacé.")
else:
    domain_report = prepare_video_pseudo_labels(
        project_root=PROJECT_ROOT,
        video_path=VIDEO_PATH,
        output_root=DOMAIN_OUTPUT_ROOT,
        teacher_model=DOMAIN_TEACHER,
        frame_stride=FRAME_STRIDE,
        max_samples=MAX_SAMPLES,
        image_size=640,
        confidence=CONFIDENCE,
        max_detections=MAX_DETECTIONS,
        source_class_id=0,
        device="mps" if torch.backends.mps.is_available() else "cpu",
    )

ensemble_report_path = ENSEMBLE_OUTPUT_ROOT / "pseudo_label_report.json"
if ensemble_report_path.exists():
    report = json.loads(ensemble_report_path.read_text())
    print("Fusion déjà présente : aucune boîte ni décision remplacée.")
else:
    report = combine_teacher_datasets(
        project_root=PROJECT_ROOT,
        # Le grand teacher adapté est prioritaire ; le nano le complète.
        source_roots=[LARGE_OUTPUT_ROOT, DOMAIN_OUTPUT_ROOT],
        output_root=ENSEMBLE_OUTPUT_ROOT,
        iou_threshold=0.5,
    )
OUTPUT_ROOT = ENSEMBLE_OUTPUT_ROOT

display(Markdown(
    f"### Résultat des teachers fusionnés\n"
    f"- Images extraites : **{report['sampled_images']:,}**\n"
    f"- Images avec au moins une boîte : **{report['frames_with_teacher_box']:,}**\n"
    f"- Images sans boîte proposée : **{report['frames_without_teacher_box']:,}**\n"
    f"- Boîtes proposées après dédoublonnage : **{report['teacher_boxes']:,}**\n"
    f"- Contributions conservées : **{report['kept_boxes_by_teacher']}**"
))

## 3. Aperçu des propositions

Rouge = proposition fusionnée des teachers. Cet aperçu sert seulement à décider si le système est suffisamment utile avant de produire la version complète.

In [ ]:
import csv

with (OUTPUT_ROOT / "manifest.csv").open(newline="") as stream:
    manifest = list(csv.DictReader(stream))

preview_indices = [round(i * (len(manifest) - 1) / 11) for i in range(12)]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for axis, index in zip(axes.flat, preview_indices):
    row = manifest[index]
    image = Image.open(PROJECT_ROOT / row["source_image"]).convert("RGB")
    draw = ImageDraw.Draw(image)
    label_path = PROJECT_ROOT / row["source_label"]
    for line in label_path.read_text().splitlines():
        _, x, y, width, height = map(float, line.split())
        left = (x - width / 2) * image.width
        top = (y - height / 2) * image.height
        right = (x + width / 2) * image.width
        bottom = (y + height / 2) * image.height
        draw.rectangle((left, top, right, bottom), outline="red", width=4)
    axis.imshow(image)
    axis.set_title(
        f"t={float(row['timestamp_seconds']):.1f}s · {row['teacher_boxes']} boîte(s)",
        fontsize=9,
    )
    axis.axis("off")
fig.suptitle("Propositions fusionnées réparties sur la vidéo")
fig.tight_layout()
plt.show()

## 4. Corriger les propositions

Les rectangles rouges sont les propositions fusionnées du grand `yolo26x` adapté et du nano adapté, pas encore une vérité terrain.

- **Valider et avancer** : toutes les boîtes affichées sont correctes ;
- **Aucun ballon** : supprimer toutes les propositions et conserver une image négative ;
- **Garder uniquement cette boîte** : conserver une seule proposition parmi plusieurs ;
- **Sauver la boîte dessinée** : remplacer toutes les propositions par ta boîte ;
- **Exclure l'image** : image inexploitable ou ambiguë.

Les décisions sont enregistrées dans ce dataset intermédiaire. Rejouer le notebook ne les efface pas.

In [ ]:
import pitchlens.data.review as review_module

review_module = importlib.reload(review_module)
store = review_module.ReviewStore(OUTPUT_ROOT)
store.progress()

In [ ]:
if "review" in globals():
    review.selector.disconnect_events()
    review.plt.close(review.figure)
    review.ui.close()
review = review_module.ReviewWidget(store)
review.display()

## 5. Progression humaine

Le dataset complet ne devra être généré qu'après avoir jugé le pilote suffisamment utile. Les images de cette vidéo resteront dans le split train ; une autre vidéo complète devra servir de validation/test de domaine.

In [ ]:
store.progress()

## 6. Benchmark contrôlé avant la version complète

Le choix d'un teacher doit être fondé sur les métriques, pas sur sa taille. Nous comparons donc le nano adapté, le `yolo26x` COCO brut et le `yolo26x` adapté sur les **670 images de validation v002**, tous à 640 px et avec le même évaluateur.

Le split test reste verrouillé : il ne sera utilisé qu'après le choix final du pipeline. Utiliser le test pour choisir le teacher ferait du test une seconde validation et rendrait son estimation trop optimiste.

In [ ]:
benchmark_path = PROJECT_ROOT / "models/reports/teacher_benchmark_v002_val/comparison.json"
if not benchmark_path.exists():
    display(Markdown(
        "Benchmark encore en cours ou non lancé. Commande reproductible :  \\n"
        "`python -m pitchlens.training.benchmark_teachers`"
    ))
else:
    benchmark = json.loads(benchmark_path.read_text())
    lines = [
        "| Modèle | mAP50 | mAP50–95 | Précision | Rappel | F1 | FP négatifs | Matériel |",
        "|---|---:|---:|---:|---:|---:|---:|---|",
    ]
    for model_name, result in benchmark["models"].items():
        fixed = result["fixed"]
        lines.append(
            f"| {model_name} | {result['map50']:.1%} | {result['map50_95']:.1%} | "
            f"{fixed['precision']:.1%} | {fixed['recall']:.1%} | {fixed['f1']:.1%} | "
            f"{fixed['fp_on_negatives']} | {result['device']} |"
        )
    display(Markdown("\n".join(lines)))
    benchmark_figure = benchmark_path.parent / "comparison.png"
    if benchmark_figure.exists():
        display(DisplayImage(filename=str(benchmark_figure)))

## 7. Même teacher, résolution d'inférence différente

Avant de réentraîner à haute résolution, nous gardons exactement le même `best.pt` entraîné à 640 px et évaluons uniquement son comportement à 640, 960 et 1280 px. Le split, les 670 images, le batch et les seuils restent identiques.

Le rappel par taille classe les vérités terrain selon la longueur de leur plus grand côté ramenée à une image de 640 px. Cela montre directement l'effet sur les très petits ballons.

In [ ]:
resolution_path = (
    PROJECT_ROOT / "models/reports/teacher_resolution_benchmark/comparison.json"
)
if not resolution_path.exists():
    display(Markdown("Le benchmark de résolution n'est pas encore disponible."))
else:
    resolution_report = json.loads(resolution_path.read_text())
    resolution_results = resolution_report["results"]
    lines = [
        "| Résolution | mAP50 | mAP50–95 | Précision | Rappel | F1 | Rappel tiny <8 px | FP négatifs | Inférence/image |",
        "|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for resolution, result in resolution_results.items():
        fixed = result["fixed"]
        tiny = result["recall_by_size"]["tiny_<8px"]["recall"]
        inference = result["speed_ms_per_image"]["inference"]
        lines.append(
            f"| {resolution} | {result['map50']:.1%} | {result['map50_95']:.1%} | "
            f"{fixed['precision']:.1%} | {fixed['recall']:.1%} | {fixed['f1']:.1%} | "
            f"{tiny:.1%} | {fixed['fp_on_negatives']} | {inference:.1f} ms |"
        )
    display(Markdown("\n".join(lines)))
    display(DisplayImage(filename=str(resolution_path.parent / "comparison.png")))

    baseline = resolution_results["640"]
    candidate = resolution_results["960"]
    tiny_640 = baseline["recall_by_size"]["tiny_<8px"]
    tiny_960 = candidate["recall_by_size"]["tiny_<8px"]
    display(Markdown(
        "**Lecture :** à 960 px, le teacher retrouve "
        f"**{tiny_960['matched'] - tiny_640['matched']} ballons tiny supplémentaires** "
        f"et gagne {(candidate['fixed']['recall'] - baseline['fixed']['recall']):+.1%} de rappel global. "
        "En revanche, ses faux positifs augmentent et son mAP50–95 baisse : "
        "960 px est donc une bonne résolution candidate pour un nouvel entraînement, "
        "mais pas encore un remplacement automatique du teacher 640 px."
    ))

## 8. Teacher réentraîné à 960 px

Le teacher 640 a été continué pendant 12 époques à 960 px sur le GPU A10G. `best.pt` correspond au meilleur état de validation, autour de l'époque 9 : les dernières époques sont conservées dans les courbes mais ne remplacent pas automatiquement le meilleur poids.

La comparaison ci-dessous distingue bien trois situations : modèle 640 utilisé à 640, simple agrandissement du modèle 640 à 960, puis véritable modèle réentraîné et utilisé à 960.

In [ ]:
highres_report_path = (
    PROJECT_ROOT / "models/reports/teacher_highres_960/training_report.json"
)
if not highres_report_path.exists():
    display(Markdown("L'entraînement haute résolution n'est pas disponible."))
else:
    highres_report = json.loads(highres_report_path.read_text())
    old_results = json.loads(resolution_path.read_text())["results"]
    new_results = highres_report["validation_after"]
    compared = [
        ("Teacher 640 @ 640", old_results["640"]),
        ("Teacher 640 @ 960", old_results["960"]),
        ("Teacher 960 @ 960", new_results["960"]),
    ]
    lines = [
        "| Expérience | mAP50 | mAP50–95 | Précision | Rappel | F1 | TP | FP | FN | FP négatifs |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for label, result in compared:
        fixed = result["fixed"]
        lines.append(
            f"| {label} | {result['map50']:.1%} | {result['map50_95']:.1%} | "
            f"{fixed['precision']:.1%} | {fixed['recall']:.1%} | {fixed['f1']:.1%} | "
            f"{fixed['tp']} | {fixed['fp']} | {fixed['fn']} | {fixed['fp_on_negatives']} |"
        )
    display(Markdown("\n".join(lines)))
    highres_root = highres_report_path.parent
    display(DisplayImage(filename=str(highres_root / "comparison_old_vs_new.png")))
    display(DisplayImage(filename=str(highres_root / "train/results.png")))

    previous = old_results["960"]["fixed"]
    current = new_results["960"]["fixed"]
    display(Markdown(
        "**Lecture :** à résolution identique de 960 px, le réentraînement ajoute "
        f"**{current['tp'] - previous['tp']} bonnes détections**, retire "
        f"**{previous['fp'] - current['fp']} fausses détections** et fait passer le F1 de "
        f"**{previous['f1']:.1%} à {current['f1']:.1%}**. "
        "Ce nouveau poids devient donc le teacher principal pour le prochain pilote vidéo."
    ))